
# Toy Problem: The Holy Grail

**You are in mysterious catacombs searching for the Holy Grail. You know it is nearby and that 
you are on the home stretch before holding the prized object in your hands. Ahead of you lies your final puzzle.**

First, you see a tile protruding from the floor, which you can step on—or not. A little further on, on the wall to your right, 
there is a painting hanging. It is crooked, and you wonder if you will resist the temptation to straighten it. 
Once past the painting, you spot a torch perched on a pillar. It is unlit, but you have a piece of iron and a flint in your possession. Finally, the Holy Grail lies before you. 

**Fortunately, you have a note in your hands explaining how to solve the puzzle.** 

So, you know the following:
1. You must activate the tile or light the torch.
1. If you step on the tile, you must straighten the painting hanging on the wall.
1. Under no circumstances should the frame be crooked and the torch be unlit.
1. If you activate the tile and light the torch, you must leave the painting crooked. 

That is all you know. Will you be able to find a combination that does not break any of the puzzle’s rules?

We will use Grover's algorithm to do so.


# Variables

First, let’s define the variables for this problem. We need to know the combination of actions required to reach the Holy Grail without breaking any rules. So we have the following variables: 

- Tile
- Frame
- Torch

The **True** (1) and **False** (0) values of these variables are labeled as follows:

|        | Variable Name |    True   |     False   |
|:------:|---------------|:---------:|:-----------:|
|  Tile  |      x₀       |   Active  | Inactive    |
| Frame  |      x₁       |   Straight   |  Crooked       |
| Torch  |      x₂       |   Lit     |   Unlit     |


# Boolean logic

not A:
$$
  ¬ A
$$ 


A or B: 
$$
  A \lor B
$$ 

A and B: 
$$
  A \land B
$$ 

If A then B
  $$
    A → B
  $$ 

Inference Rule
  $$
    A → B = \neg A \lor B.
  $$ 

De Morgan's Rule
  $$
    A \land B = \neg(¬ A ∨  ¬ B)
  $$ 



# The conditions 

We want to satisfy all conditions. We can describe such a state with a function of the form 
$$
f(x_0, x_1, x_2) = (cond. 1) ∧ (cond. 2) ∧ (cond. 3) ∧ (cond. 4) 
$$

Now, for each condition in turn, we have:

1. You must activate the tile or light the torch.
<!-- # The tile must be activated or the torch must be lit.
# (x_0 ⌵ x_2) -->
  $$
    x_0 \lor x_2.
  $$ 

2. If you step on the tile, you must straighten the picture hanging on the wall. Use the inference rule.
<!-- # If the tile is activated, the frame must be straight
# x0 -> x1 = (¬x_0 ⌵ x_1) -->
  $$
    x_0 → x_1 = \neg x_0 \lor x_1.
  $$ 
    

3. Under no circumstances should the frame be crooked and the torch be off. Use De Morgan's Rule.
<!-- # The frame must be straight or the torch must be lit
# (x_1 ⌵ x_2) -->
  $$
    \neg (\neg x_1 \land \neg x_2) = \neg (\neg ( x_1 \lor x_2))  = x_1 \lor x_2.
  $$

4. If you activate the tile and light the torch, you must leave the frame crooked. Use inference rule, then De Morgan's.
<!-- # If the tile is activated and the torch is lit, then the frame must be crooked
#  x0 ^ x2 -> ¬x1 =  (¬x0 ⌵ ¬x2 ⌵ ¬x1)  -->
  $$
    (x_0 \land x_2) → \neg x_1 = ¬(x_0 \land x_2) \lor \neg x_1 = ¬(¬(¬x_0 \lor ¬x_2)) \lor \neg x_1 = \neg x_0 \lor \neg x_2 \lor \neg x_1.
  $$ 


# The logical formula

The logical formula for the problem, in conjunctive normal form, is therefore given by 
$$
f(x) = f(x_0, x_1, x_2) = (x_0 \lor x_2) \land (\neg x_0 \lor x_1) \land (x_1 \lor x_2) \land (\neg x_0 \lor \neg x_2 \lor \neg x_1)
$$



**Now let’s have some fun coding Grover!**

***

# Import the necessary libraries

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from qiskit.circuit.library import XGate, ZGate
from qiskit_aer import Aer
from qiskit import transpile

# Define the propositions 

A clause of the form

$
    (x_0 \lor x_1 \lor \neg x_2)
$

is written as: 
```
clause = {‘x0’ : True, ‘x1’ : True, ‘x2’ : False}
````


In [ ]:
clauses = []

# The tile must be activated or the torch must be lit.
# (x_0 ⌵ x_2)
clauses.append({"x0": True, "x2": True})

# If the tile is activated, the frame must be straight
# (¬x₀ ∨ x₁)
clauses.append({"x0": False, "x1": True})

# The frame must not be crooked and the torch must not be off.
# (x₁ ∨ x₂)
clauses.append({"x1": True, "x2": True})

# If the tile is activated and the torch is on, then the frame must be crooked
# (¬x₀ ∨ ¬x₂ ∨ ¬x₁)
clauses.append({"x0": False, "x2": False, "x1": False})

In [ ]:
from itertools import product

def eval_clause(clause, assignment):
    return any((assignment[var] == 1) if is_positive else (assignment[var] == 0)
               for var, is_positive in clause.items())

def eval_formula(clauses, assignment):
    return all(eval_clause(clause, assignment) for clause in clauses)

classical_solutions = []
for x0, x1, x2 in product([0, 1], repeat=3):
    assignment = {"x0": x0, "x1": x1, "x2": x2}
    if eval_formula(clauses, assignment):
        classical_solutions.append((x0, x1, x2))

print("Classical SAT solutions (x0, x1, x2):", classical_solutions)
print("Equivalent measured bitstrings (x2x1x0):", [f"{x2}{x1}{x0}" for x0, x1, x2 in classical_solutions])

# Identify the known facts of the problem

In [ ]:
# x0: tile, x1: array, x2: torch, x3: path
nb_variables = 3
# 3 clauses
nb_clauses = len(clauses)

# Creating a function to convert a disjunction into a circuit

We will define, step by step, a function that takes a disjunction as input and converts it into a quantum circuit. 

For example, the formula: 

$
    c = x_0 \lor x_1 \lor \neg x_2
$

would be transformed into the following series of gates: 

<img src="./disj_circuit.png" alt= “” width="15%" height="15%">

Each disjunction is transformed into a circuit containing as many qubits as there are variables in the formula,
plus an auxiliary qubit associated with the clause in question. The circuit is then converted into a gate that can be
added to a circuit (such as the oracle), by specifying the correct qubits. 

## 1. Control State of Each Clause

Let’s start by creating a function that returns the correct control state for a given disjunctive clause.
For example, the clause 
$$
c_1 = x_0 \lor x_1 \lor \neg x_2
$$
uses qubit 0 (variable x_0), qubit 1 (variable x_1), and qubit 2 (variable x_2).

The NOT function on variable x_2 inverts the control state of qubit 2. This function should therefore return the control state “100”. 

As a second example, the clause
$$
c_2 = x_0 \lor \neg x_2
$$
uses qubit 0 (variable x_0) and qubit 2 (variable x_2).

The negation on variable x_2 inverts the control state of qubit 2. This function must therefore return the control state “10”.

**Take the time to write a function that performs this operation.**


In [ ]:
def get_disjunction_control_state(clause):
    ctrl_state = ""
    for var_name in clause:
        if clause[var_name]:      # positive literal like x₀
            ctrl_state += "0"     # MCX fires when x₀ = 0 (literal is false)
        else:                     # negated literal like ¬x₀
            ctrl_state += "1"     # MCX fires when x₀ = 1 (¬x₀ is false)
    return ctrl_state[::-1]

## 2. Constructing a Multi-controlled Gate

Once the control state of a clause has been determined, we can construct a **multi-controlled quantum gate** that will be activated only when the control qubits correspond to that state.

In **Qiskit**, you can convert a gate to a multi-controlled version using the `.control()` method. You must specify:
- the **number of control qubits**
- the **control state**

For example, you can create a controlled gate as follows:

```python
    ZGate().control(num_ctrl_qubits=2, ctrl_state="01")
``` 

**Which gate do we want to control here? With how many qubits? Take the time to write the following function.**

In [ ]:
def get_multi_control_gate(nb_disj_variables, ctrl_state):
    return XGate().control(num_ctrl_qubits=nb_disj_variables, ctrl_state=ctrl_state)

## 3. Converting a Disjunction into a Quantum Gate

Now let’s combine all these blocks to convert the disjunction into a quantum gate. 

**Complete the following function.**

In [ ]:
# Transform a disjunction into a gate
def logical_disjunction_to_gate(disj_clause):
    nb_disj_variables = len(disj_clause)
    nb_qubits = nb_disj_variables + 1
    disj_qc = QuantumCircuit(nb_qubits)
    ctrl_state = get_disjunction_control_state(disj_clause)
    mc_xgate = get_multi_control_gate(nb_disj_variables, ctrl_state)
    qubits = list(range(nb_qubits))
    disj_qc.append(mc_xgate, qubits)
    disj_qc.x(qubits[-1])
    disj_gate = disj_qc.to_gate(label="mcx")
    return disj_gate

**Let's see how we can use this function.**

In [ ]:
clause_test = {"x0": True, "x1": True, "x2": False}

test_qc = QuantumCircuit(4)
disj_gate = logical_disjunction_to_gate(clause_test)
test_qc.append(disj_gate, test_qc.qubits)
test_qc.decompose().draw("mpl")

## 3. Retrieving the Qubits of a Disjunction Clause

To apply the mcx quantum gate to our Grover circuit, we must identify **the qubits involved in the clause**.  
The following function therefore constructs the list of qubits corresponding to the variables in the clause, as well as the **qubit associated with the clause**.

The clause is given in the form of a dictionary, for example:

```
{“x2”: True, ‘x0’: True, “x1”: True}
```

The keys of the dictionary are the **names of the variables** (`x0`, `x1`, etc.).  
The first step is therefore to convert these names into **integer indices** using the function:

```python
variable_name_to_int(var_name)
```

For example:

| Variable | Index |
| -------- | ----- |
| x0       | 0     |
| x1       | 1     |
| x2       | 2     |

This function is provided to you.


In [ ]:
# Function that return an int related to the variable name.
def variable_name_to_int(var_name):
    if len(var_name) == 2 and var_name[1].isdigit():
        return int(var_name[1])
    else:
        raise ValueError("Input must be a string of format 'xi' where i is a digit")

These indices allow the following function to access the correct qubits in the `var_qubits` list, which contains the qubits associated with the variables.

This function performs the following steps:

1. iterate through the variables in the clause,
2. convert each variable name to an **integer index**,
3. retrieve the **corresponding qubit** from `var_qubits`,
4. append the **clause qubit** (`clause_qubit`) to the end.

For example, if:

* `disj_clause = {“x2”: True, “x0”: True}`
* `var_qubits = [q0, q1, q2]`
* `clause_qubit = c0`

the function will return:

```
[q2, q0, c0]
```

The first three qubits correspond to the **clause variables**, and the last one is the **target qubit representing the clause** in the oracle.


In [ ]:
def get_disjunction_qubits(disj_clause, clause_qubit, var_qubits):
    variables = []
    for var_name in disj_clause:
        variables.append(variable_name_to_int(var_name=var_name))

    disjunction_qubits = []

    for var in variables:
        disjunction_qubits.append(var_qubits[var])

    disjunction_qubits.append(clause_qubit)

    return disjunction_qubits

# Building the Oracle

The following cells build the oracle for the Holy Grail problem. You will need to build a similar oracle for the Package problem. 

In [ ]:
# Create quantum registers for variables and clauses
var_qubits = QuantumRegister(nb_variables, name="x")
clause_qubits = QuantumRegister(nb_clauses, name="c")

In [ ]:
# Build the clause circuit
clauses_circuit = QuantumCircuit(var_qubits, clause_qubits)

# Add each disjunction clause as a gate:
for i in range(nb_clauses):
    gate = logical_disjunction_to_gate(clauses[i])
    c_qubits = get_disjunction_qubits(clauses[i], clause_qubits[i], var_qubits)
    clauses_circuit.append(gate, c_qubits)

# Display the circuit:
clauses_circuit.decompose(gates_to_decompose=["mcx"], reps=2).draw(output="mpl")

In [ ]:
# Build the oracle
oracle_circuit = QuantumCircuit(var_qubits, clause_qubits)

# Add the clauses circuit
# --- Converting it to a gate is only useful for display purposes later on ---
oracle_circuit.append(clauses_circuit.to_gate(label="clauses_circuit"), clauses_circuit.qubits)

# Add the Z multi-control gate
mc_z_gate = ZGate().control(nb_clauses - 1)
oracle_circuit.append(mc_z_gate, clause_qubits)

# Add the inverse of the clause circuit
oracle_circuit.append(clauses_circuit.reverse_ops().to_gate(label="clauses_circuit"), oracle_circuit.qubits)

# Print the circuit
oracle_circuit.decompose(gates_to_decompose=["clauses_circuit", "mcx"], reps=2).draw(output="mpl")

# Building the diffuser

Here is the diffuser circuit.

In [ ]:
# Build the diffuser circuit
diffuser_circuit = QuantumCircuit(var_qubits)

# Add H and X gates for each qubit in the diffuser
diffuser_circuit.h(range(nb_variables))
diffuser_circuit.x(range(nb_variables))

# Add a multi-controlled Z
mc_z_gate = ZGate().control(nb_variables - 1)
diffuser_circuit.append(mc_z_gate, var_qubits)

# Add X and H gates for each qubit in the diffuser
diffuser_circuit.x(range(nb_variables))
diffuser_circuit.h(range(nb_variables))

# Display the circuit
diffuser_circuit.draw(output="mpl")

# Building the Grover Circuit

Here is the Grover circuit.

In [ ]:
# Build the Grover circuit
c_bits = ClassicalRegister(nb_variables)
grover_circuit = QuantumCircuit(var_qubits, clause_qubits, c_bits)

# Add H gates for each variable
grover_circuit.h(range(nb_variables))

# Identify the number of iterations
nb_iterations = 1  # Play with the number of iterations to see its effect

# Add as many oracles and diffusers as the number of iterations
for it in range(nb_iterations):
    grover_circuit.append(oracle_circuit.to_gate(label="oracle"), grover_circuit.qubits)
    grover_circuit.barrier(grover_circuit.qubits)
    grover_circuit.append(diffuser_circuit.to_gate(label="diffusor"), grover_circuit.qubits[0:nb_variables])

# Add measurements for evaluating the circuit
grover_circuit.measure(var_qubits, c_bits)

# Display the circuit
grover_circuit.decompose(gates_to_decompose=["oracle", "clauses_circuit", "diffusor", "mcx"], reps=3).draw(
    output="mpl", scale=0.8
)

# Measurement of the solution

Execute the circuit and measure the qubits to see which states were amplified.

In [ ]:
# Prepare a simulation to run and measure the solution
simulator = Aer.get_backend("qasm_simulator")

# Transpile the circuit, run it, and get the counts of the solutions
new_circuit = transpile(grover_circuit, backend=simulator)
job = simulator.run(new_circuit, shots=10000)
result = job.result()
counts = result.get_counts()

print(counts)

In [ ]:
# Quick consistency check: Grover vs classical SAT
expected_bits = set()
if "classical_solutions" in globals():
    expected_bits = {f"{x2}{x1}{x0}" for x0, x1, x2 in classical_solutions}
else:
    from itertools import product
    def _eval_clause(clause, assignment):
        return any((assignment[var] == 1) if is_positive else (assignment[var] == 0)
                   for var, is_positive in clause.items())
    def _eval_formula(clauses, assignment):
        return all(_eval_clause(clause, assignment) for clause in clauses)
    for x0, x1, x2 in product([0, 1], repeat=3):
        assignment = {"x0": x0, "x1": x1, "x2": x2}
        if _eval_formula(clauses, assignment):
            expected_bits.add(f"{x2}{x1}{x0}")

top_measured = {k for k, _ in sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:len(expected_bits)]}

print("Expected SAT bitstrings:", sorted(expected_bits))
print("Top measured bitstrings:", sorted(top_measured))

missing = sorted(expected_bits - top_measured)
unexpected = sorted(top_measured - expected_bits)

if not missing and not unexpected:
    print("✅ PASS: Grover results match classical SAT solutions.")
else:
    print("❌ FAIL: mismatch detected.")
    print("Missing expected:", missing)
    print("Unexpected extra:", unexpected)

In [ ]:
# Plot the probability histogram
plot_histogram(counts)

|         | Variable Name |    True    |     False    |
|:-------:|-----------------|:----------:|:-----------:|
|  Tile  |      $x_0$      |   Enabled  | Disabled |
| Table |      $x_1$      |  Straight     |  Crooked     |
| Torch  |      $x_2$      | On    |   Off   |

You can see that you have three options:
1. Activate the tile, straighten the frame, and leave the torch off;
2. Avoid the tile, leave the frame crooked, and turn on the torch;
3. Avoid the tile, straighten the frame, and turn on the torch.